In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

rng = np.random.default_rng(3)
n = 800
df = pd.DataFrame({
    "Brand": rng.choice(["Maruti", "Hyundai", "Honda", "Toyota", "Ford"], n),
    "Year": rng.integers(2008, 2025, n),
    "KmDriven": rng.integers(5000, 200000, n),
    "FuelType": rng.choice(["Petrol", "Diesel", "CNG"], n),
    "Transmission": rng.choice(["Manual", "Automatic"], n),
    "EngineCC": rng.choice([800, 1000, 1200, 1500, 2000], n),
})
brand_value = {"Maruti": 0, "Hyundai": 30000, "Honda": 60000, "Toyota": 100000, "Ford": 20000}
df["Price"] = (150000 + (df["Year"] - 2008) * 25000 - df["KmDriven"] * 0.8
               + df["EngineCC"] * 150 + df["Brand"].map(brand_value)
               + (df["Transmission"] == "Automatic") * 80000
               + (df["FuelType"] == "Diesel") * 50000
               + rng.normal(0, 20000, n)).clip(lower=50000)
df.to_csv("car_price.csv", index=False)

df = pd.read_csv("car_price.csv")
print(df.head(), "\n")

X = df.drop("Price", axis=1)
y = df["Price"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

categorical = ["Brand", "FuelType", "Transmission"]
preprocess = ColumnTransformer(
    [("cat", OneHotEncoder(handle_unknown="ignore"), categorical)],
    remainder="passthrough")

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42),
}
best_pipe = None
for name, reg in models.items():
    pipe = Pipeline([("prep", preprocess), ("model", reg)]).fit(X_train, y_train)
    pred = pipe.predict(X_test)
    print(f"{name}: R2 = {r2_score(y_test, pred):.4f}, "
          f"MAE = {mean_absolute_error(y_test, pred):.0f}, "
          f"RMSE = {np.sqrt(mean_squared_error(y_test, pred)):.0f}")
    best_pipe = pipe

new_car = pd.DataFrame([{"Brand": "Honda", "Year": 2020, "KmDriven": 40000,
                         "FuelType": "Petrol", "Transmission": "Manual", "EngineCC": 1200}])
print("\nPredicted price of the new car: Rs.", round(best_pipe.predict(new_car)[0]))

plt.scatter(y_test, best_pipe.predict(X_test), s=10)
plt.plot([y.min(), y.max()], [y.min(), y.max()], "r--")
plt.xlabel("Actual price")
plt.ylabel("Predicted price")
plt.title("Car Price Prediction (Random Forest)")
plt.show()